#use llamafactory environment

In [1]:
from datasets import load_dataset
import pandas as pd
import json


In [2]:
#load training set of hellaswag
dataset = load_dataset("allenai/ai2_arc", name="ARC-Challenge", split="train")

#print dataset info
print("Dataset info:")
print(dataset)

print("# samples:", len(dataset)) #should be 39k
n_samples = len(dataset)

#convert to dataframe
df = dataset.to_pandas()
df.head()


Dataset info:
Dataset({
    features: ['id', 'question', 'choices', 'answerKey'],
    num_rows: 1119
})
# samples: 1119


,id,question,choices,answerKey
0,Mercury_SC_415702,George wants to warm his hands quickly by rubb...,"{'text': ['dry palms', 'wet palms', 'palms cov...",A
1,MCAS_2009_5_6516,Which of the following statements best explain...,"{'text': ['The refrigerator door is smooth.', ...",B
2,Mercury_7233695,A fold observed in layers of sedimentary rock ...,"{'text': ['cooling of flowing magma.', 'conver...",B
3,Mercury_7041615,Which of these do scientists offer as the most...,"{'text': ['worldwide disease', 'global mountai...",D
4,Mercury_7041860,A boat is acted on by a river current flowing ...,"{'text': ['west', 'east', 'north', 'south'], '...",B


In [3]:
# Add an 'answer' column by matching answerKey to the corresponding choice text.
def extract_answer(row):
    labels = row["choices"]["label"]
    texts = row["choices"]["text"]
    label_to_text = dict(zip(labels, texts))
    return label_to_text.get(row["answerKey"], None)

df["answer"] = df.apply(extract_answer, axis=1)
df.head(100)

,id,question,choices,answerKey,answer
0,Mercury_SC_415702,George wants to warm his hands quickly by rubb...,"{'text': ['dry palms', 'wet palms', 'palms cov...",A,dry palms
1,MCAS_2009_5_6516,Which of the following statements best explain...,"{'text': ['The refrigerator door is smooth.', ...",B,The refrigerator door contains iron.
2,Mercury_7233695,A fold observed in layers of sedimentary rock ...,"{'text': ['cooling of flowing magma.', 'conver...",B,converging of crustal plates.
3,Mercury_7041615,Which of these do scientists offer as the most...,"{'text': ['worldwide disease', 'global mountai...",D,impact of an asteroid created dust that blocke...
4,Mercury_7041860,A boat is acted on by a river current flowing ...,"{'text': ['west', 'east', 'north', 'south'], '...",B,east
...,...,...,...,...,...
95,Mercury_7182140,Cellular respiration results in the production...,"{'text': ['oxygen and energy', 'glucose and gl...",D,carbon dioxide and water
96,Mercury_LBS10002,The following mathematical expressions represe...,"{'text': ['1.0 x 10^3', '1.0 x 10^4', '1.0 x 1...",C,1.0 x 10^-3
97,MSA_2015_8_37,Data in tables may also be presented in graphs...,{'text': ['the distance of the planets from th...,D,the percent of various materials in solid waste
98,Mercury_7200848,A plant that grows red flowers was crossed wit...,{'text': ['The offspring experienced a genetic...,C,The genes for flower color exhibited incomplet...


In [5]:
### create json file

#format data for sft
data = []

for idx, row in df.iterrows():
    item = {
        "instruction": row["question"],
        "input": "",
        "output": row["answer"]
    }
    data.append(item)
    #track progress
    if idx % 10000 == 0:
        print(f"Processed {idx} rows")
    if idx == (n_samples - 1):
        print(f"Processed {idx} rows (last row)")

#save to JSON file
with open("data/arc_challenge.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print("Complete!")

Processed 0 rows
Processed 1118 rows (last row)
Complete!
